# Training the encoder
## Notes
I am testing three encoder architectures to learn the mapping from images to synthetic neural responses:
- A simple CNN (with batch norm, max pooling, dropout)
- A ResNet18 pretrained on ImageNet
- A simple CNN like the first **plus** skip connections

All of them expect a number of output neurons equal to the number of synthetic neurons (100) in the given dataset.

I am testing different learning rates and batch sizes. 
Optimiser is Adam, loss MSE
I am running 5-fold cross validation thorugh 40 epochs.
I am developing using Pytorch lightnning (especially Trainer, callbacks, LightningDataModule)and mlflow.

Also:
- mixed precision (with Pytorch Lightning - to reduce memory footprint during model training)


Let's look at the training of Resnet as an encoder.

In [2]:
import mlflow
from pathlib import Path


path_to_mlflow_runs = Path("/ceph/margrie/laura/neurodecoders/mlruns/")

# load all runs
mlflow.set_tracking_uri(f"file://{path_to_mlflow_runs}")
experiments = mlflow.search_experiments()
experiments

[<Experiment: artifact_location='file:///ceph/margrie/laura/neurodecoders/mlruns/261857491297849044', creation_time=1756323014084, experiment_id='261857491297849044', last_update_time=1756323014084, lifecycle_stage='active', name='fourth_run/resnet_encoder_comparison', tags={}>,
 <Experiment: artifact_location='file:///ceph/margrie/laura/neurodecoders/mlruns/325587642601194576', creation_time=1756317523977, experiment_id='325587642601194576', last_update_time=1756317523977, lifecycle_stage='active', name='fourth_run/simple_encoder_comparison', tags={}>,
 <Experiment: artifact_location='file:///ceph/margrie/laura/neurodecoders/mlruns/406052450740737059', creation_time=1756311107482, experiment_id='406052450740737059', last_update_time=1756311107482, lifecycle_stage='active', name='fourth_run/skip_encoder_comparison', tags={}>,
 <Experiment: artifact_location='file:///ceph/margrie/laura/neurodecoders/mlruns/577499894404631378', creation_time=1756301142377, experiment_id='5774998944046313

In [3]:
#  get the one with name fourth_run/resnet_encoder_comparison
name = "fourth_run/resnet_encoder_comparison"
experiment = [exp for exp in experiments if exp.name == name][0]
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.system/disk_usage_megabytes,metrics.system/system_memory_usage_megabytes,metrics.system/disk_available_megabytes,metrics.system/gpu_0_utilization_percentage,...,params.dataset_input_shape,params.dataset_n_neurons,params.dataset_sta_patch_height,params.dataset_dataset_filename,params.dataset_sta_pattern,tags.mlflow.source.git.commit,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.source.name,tags.mlflow.user
0,d51244febbe8491b85a58d6b564425d8,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:50:33.903000+00:00,2025-08-27 19:54:03.632000+00:00,18990.7,14561.1,1893174.1,0.0,...,"(10000, 1, 224, 224)",100,70,"synthdata_dataset-mnist_sta-periodic_patterns,...",periodic_patterns,810cd81e821a31c270fba000ec2107625c46cf9f,LOCAL,lr5e-3_bs16_epochs40_task24_fold_3,neurodecoders/encoder/mlflow_training.py,lporta
1,bb6f867e622949f8a6d65281a415c151,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:50:30.663000+00:00,2025-08-27 19:53:21.445000+00:00,18990.8,25647.6,1893174.1,0.0,...,"(10000, 1, 224, 224)",100,70,"synthdata_dataset-mnist_sta-periodic_patterns,...",periodic_patterns,810cd81e821a31c270fba000ec2107625c46cf9f,LOCAL,lr5e-3_bs32_epochs40_task25_fold_3,neurodecoders/encoder/mlflow_training.py,lporta
2,aa671e65f6df49d88785243dc7817709,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:48:29.150000+00:00,2025-08-27 19:49:52.860000+00:00,20266.6,74475.0,426565.1,0.0,...,"(10000, 1, 224, 224)",100,70,"synthdata_dataset-mnist_sta-periodic_patterns,...",periodic_patterns,810cd81e821a31c270fba000ec2107625c46cf9f,LOCAL,lr5e-3_bs64_epochs40_task26_fold_3,neurodecoders/encoder/mlflow_training.py,lporta
3,5033243bec24428e96e0b24e484e2147,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:47:40.661000+00:00,2025-08-27 19:50:29.976000+00:00,18990.8,21586.3,1893174.1,0.0,...,"(10000, 1, 224, 224)",100,70,"synthdata_dataset-mnist_sta-periodic_patterns,...",periodic_patterns,810cd81e821a31c270fba000ec2107625c46cf9f,LOCAL,lr5e-3_bs32_epochs40_task25_fold_2,neurodecoders/encoder/mlflow_training.py,lporta
4,995c0ead11624d1ebd1928da42146c80,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:47:02.070000+00:00,2025-08-27 19:48:28.525000+00:00,20266.6,72476.8,426565.1,0.0,...,"(10000, 1, 224, 224)",100,70,"synthdata_dataset-mnist_sta-periodic_patterns,...",periodic_patterns,810cd81e821a31c270fba000ec2107625c46cf9f,LOCAL,lr5e-3_bs64_epochs40_task26_fold_2,neurodecoders/encoder/mlflow_training.py,lporta
5,e2da836f328d4d5e815ff8974b17615e,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:47:02.023000+00:00,2025-08-27 19:50:33.211000+00:00,18990.8,21594.2,1893174.1,0.0,...,"(10000, 1, 224, 224)",100,70,"synthdata_dataset-mnist_sta-periodic_patterns,...",periodic_patterns,810cd81e821a31c270fba000ec2107625c46cf9f,LOCAL,lr5e-3_bs16_epochs40_task24_fold_2,neurodecoders/encoder/mlflow_training.py,lporta
6,c0251a8346cb4cc6ad69740d5f158cc6,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:46:26.938000+00:00,2025-08-27 19:49:05.122000+00:00,19933.5,38902.4,1892231.4,0.0,...,"(10000, 1, 224, 224)",100,70,"synthdata_dataset-mnist_sta-periodic_patterns,...",periodic_patterns,810cd81e821a31c270fba000ec2107625c46cf9f,LOCAL,lr1e-3_bs64_epochs40_task23_fold_3,neurodecoders/encoder/mlflow_training.py,lporta
7,43690d5f06124941849c050166d29598,261857491297849044,FINISHED,file:///ceph/margrie/laura/neurodecoders/mlrun...,2025-08-27 19:45:14.904000+00:00,2025-08-27 19:47:01.477000+00:00,20266.6,70440.1,426565.1,0.0,...,"(10000, 1, 224, 224)",100,70,"synthdata_dataset-mnist_sta-periodic_patterns,...",periodic_patterns,810cd81e821a31c270fba000ec2107625c46cf9f,LOCAL,lr5e-3_bs64_epochs40_task26_fold_1,neurodecoders/enco

In [5]:
# let's only choose those that have status FINISHED and take the top 10 by val_loss and print train loss and val loss
runs = runs[runs["status"] == "FINISHED"]
runs = runs.sort_values(by=["metrics.val_loss"])
top_10_runs = runs.head(10)
for index, run in top_10_runs.iterrows():
    print(f"Train Loss: {run['metrics.train_loss']:2f}, Val Loss: {run['metrics.val_loss']:2f}, run_name: {run['tags.mlflow.runName']}")

Train Loss: 3.789138, Val Loss: 16.758606, run_name: lr5e-4_bs16_epochs40_task18_fold_2
Train Loss: 2.760064, Val Loss: 16.960333, run_name: lr1e-3_bs16_epochs40_task21_fold_2
Train Loss: 6.893279, Val Loss: 17.039955, run_name: lr5e-4_bs32_epochs40_task19_fold_2
Train Loss: 3.612371, Val Loss: 17.067032, run_name: lr1e-3_bs32_epochs40_task22_fold_2
Train Loss: 1.915294, Val Loss: 17.294289, run_name: lr1e-3_bs16_epochs40_task21_fold_3
Train Loss: 6.079440, Val Loss: 17.366009, run_name: lr5e-4_bs64_epochs40_task20_fold_3
Train Loss: 1.720755, Val Loss: 17.529062, run_name: lr5e-4_bs16_epochs40_task18_fold_3
Train Loss: 11.213946, Val Loss: 17.580606, run_name: lr5e-4_bs64_epochs40_task20_fold_2
Train Loss: 1.890170, Val Loss: 17.666798, run_name: lr1e-3_bs32_epochs40_task22_fold_3
Train Loss: 6.793271, Val Loss: 17.773151, run_name: lr1e-3_bs64_epochs40_task23_fold_2


After examining the results, I decided to move forward with pretrained ResNet18 with learning rate 1e-3 and batch size 32. I am going to move on with the following modifications:
- with a new dataset with higher spatial frequency components (perlin noise patterns)
- by separating training and test sets into different folders
- by testing:
    - learning rate schedulers (ReduceLROnPlateau, CosineAnnealingLR)
    - weight decay (1e-5, 1e-4)
    - different optimisers (AdamW, SGD with momentum)
    - warmup
- I should still be doing cross-validation (5 folds) to do hyperparameter search

A small note: I think I was doing CV wrong in the previous experiments. I was doing 3-fold CV but starting training on the next fold with the model weights from the previous fold. This is not correct, as each fold should be independent. I might need to re-run hyperparam search for learning dates ad batch size.